# Reproducción completa en Google Colab — `causal-ga-dml`

Descubrimiento causal con algoritmos genéticos + Double Machine Learning.
Autor: Diego Alonso Córdova Ayala · UNI, Perú · Taller DES-304.

**Modo de uso:** menú *Entorno de ejecución → Ejecutar todo*. Las celdas corren en orden
y al final se muestran **las 6 tablas y las 4 figuras** del proyecto de tesis y del artículo.

El experimento completo (30 réplicas) tarda ~12–20 min en una CPU de Colab. No requiere GPU
ni descargar datos: la red ALARM viene dentro de la librería `pgmpy`.

## 1. Traer el código desde GitHub

In [ ]:
REPO = 'https://github.com/diego-555-dmg/causal-ga-dml.git'   # <-- si tu repositorio tiene otra URL, cámbiala aquí
import os
!rm -rf /content/causal-ga-dml
!git clone $REPO /content/causal-ga-dml
PROJECT_DIR = '/content/causal-ga-dml'
assert os.path.exists(f'{PROJECT_DIR}/pyproject.toml'), 'No se clonó bien el repositorio.'
%cd $PROJECT_DIR
print('Proyecto en:', PROJECT_DIR)

### Alternativa sin GitHub (subir ZIP)

Si prefieres no usar GitHub, **omite la celda de arriba**, ejecuta esta, y sube el
`causal-ga-dml.zip` (comprime la carpeta en tu PC con clic derecho → *Enviar a → Carpeta comprimida*).

In [ ]:
# --- Descomenta TODO este bloque solo si vas por ZIP en vez de GitHub ---
# from google.colab import files
# import zipfile, glob, os
# subidos = files.upload()
# with zipfile.ZipFile(next(iter(subidos))) as z: z.extractall('/content')
# PROJECT_DIR = os.path.dirname(glob.glob('/content/**/pyproject.toml', recursive=True)[0])
# %cd $PROJECT_DIR
# print('Proyecto en:', PROJECT_DIR)

## 2. Instalar dependencias (versiones exactas)

Verás una advertencia roja de `pip` diciendo que `google-colab` o `numba` querrían otras
versiones de pandas/numpy: **es normal e inofensiva**, esos paquetes no intervienen en el pipeline.
Si Colab ofrece *Restart runtime*, púlsalo y vuelve a ejecutar **solo** esta celda.

In [ ]:
import glob, os
if 'PROJECT_DIR' not in globals():
    PROJECT_DIR = os.path.dirname(glob.glob('/content/**/pyproject.toml', recursive=True)[0])
%cd $PROJECT_DIR
!pip install -q -r requirements.txt
print('Dependencias instaladas.')

## 3. Registrar el entorno (hardware y versiones de esta sesión de Colab)

In [ ]:
import sys, json
sys.path.insert(0, f'{PROJECT_DIR}/src')
from causal_ga_dml.config import environment_report
print(json.dumps(environment_report(), indent=2, ensure_ascii=False))

## 4. Verificación rápida (~30–60 s)

Corre el pipeline una vez con la semilla 42 para confirmar que todo funciona.

In [ ]:
!python scripts/run_single.py --config configs/default.yaml

## 5. Experimento completo: 30 réplicas (~12–20 min)

Reproduce todas las cifras del artículo y de la tesis. Es **resumible**: si la sesión se corta,
vuelve a ejecutar esta misma celda y continúa donde quedó. Genera el JSON de resultados y las 4 figuras.

In [ ]:
!python scripts/run_multiseed.py --reps 30

### ¿Poco tiempo? Prueba rápida (~1 min)

Solo para ver el flujo de principio a fin; **no** reproduce las cifras publicadas.
Si usas esta celda, salta la del experimento completo de arriba.

In [ ]:
# !python scripts/run_replicas.py --config configs/quick.yaml --reps 3 && \
#  python scripts/aggregate.py --config configs/quick.yaml

## 6. Mostrar TODAS las tablas del documento

Esta celda reconstruye por sí misma —a partir del JSON de resultados— **las 6 tablas** tal como
figuran en el proyecto de tesis. La Tabla 1 es la comparación conceptual de enfoques; las Tablas 2–6
salen de los datos del experimento.

In [ ]:
import glob, json
import pandas as pd
from IPython.display import display, Markdown

ruta = sorted(glob.glob(f'{PROJECT_DIR}/results/multiseed_*.json'))[-1]
data = json.load(open(ruta, encoding='utf-8'))
R, ENT, CFG = data['resumen'], data['entorno'], data['configuracion']
NOM = {'AG':'Algoritmo genético','Hill-Climbing':'Hill-Climbing (BIC)',
       'PC':'PC (Fisher-z)','Orden aleatorio':'Orden aleatorio (ablación)'}
ms = lambda b, d=3: f"{b['media']:.{d}f} ± {b['de']:.{d}f}"
cm = lambda x, d=3: f'{x:.{d}f}'

def mostrar(titulo, df):
    display(Markdown(f'### {titulo}'))
    display(df)

t1 = pd.DataFrame([
  ['Restricciones (PC)','Pruebas de independencia','Fundamento teórico explícito','Errores en cascada; sensibilidad al orden'],
  ['Score (Hill-Climbing, A*)','Optimización de score penalizado','Criterio global comparable','Óptimos locales; NP-dificultad'],
  ['Híbrida (MMHC)','Restricción + score','Compromiso exactitud/costo','Hereda debilidades de ambas'],
  ['Continua (NOTEARS)','Aciclicidad diferenciable','Optimización por gradiente','Vulnerable a artefactos de datos'],
  ['Evolutiva (AG)','Búsqueda poblacional global','Escapa de óptimos locales','Mayor costo computacional'],
], columns=['Familia','Mecanismo','Fortaleza','Limitación principal'])

t2 = pd.DataFrame([
  ['Datos','Red / observaciones',f"{CFG['data']['network']} / {CFG['data']['n_samples']}"],
  ['Datos','Tratamiento / resultado',f"{CFG['data']['treatment']} / {CFG['data']['outcome']}"],
  ['Score','Penalización BIC / regularización',f"{CFG['score']['penalty']} / {CFG['score']['ridge']}"],
  ['AG','Población / generaciones',f"{CFG['ga']['population_size']} / {CFG['ga']['n_generations']}"],
  ['AG','Grado de entrada máximo',CFG['ga']['max_indegree']],
  ['AG','p. cruce / p. mutación',f"{CFG['ga']['p_crossover']} / {CFG['ga']['p_mutation']}"],
  ['AG','Torneo / elitismo',f"{CFG['ga']['tournament_size']} / {CFG['ga']['elitism']}"],
  ['Double ML','Particiones / árboles / prof.',f"{CFG['dml']['n_splits']} / {CFG['dml']['n_estimators']} / {CFG['dml']['max_depth']}"],
  ['Diseño','Réplicas / semilla maestra',f"{CFG['n_replications']} / {CFG['seed']}"],
], columns=['Componente','Parámetro','Valor'])

t3 = pd.DataFrame([{'Método':NOM[m],'F1 esqueleto':ms(b['estructura']['f1_esqueleto']),
   'Recall esqueleto':ms(b['estructura']['recall_esqueleto']),
   'Precisión esqueleto':ms(b['estructura']['precision_esqueleto']),
   'F1 dirigido':ms(b['estructura']['f1_dirigido']),'SHD':ms(b['estructura']['SHD'],1)}
   for m,b in R['por_metodo'].items()])

t4 = pd.DataFrame([{'Método':NOM[m],'Cobertura de confusores':cm(b['cobertura_confusores']['media']),
   'Contaminación por descendientes':cm(b['tasa_contaminacion_descendientes']*100,1)+' %',
   '|Sesgo| vs. oráculo':cm(b['sesgo_absoluto_vs_oraculo']['media'],4)}
   for m,b in R['por_metodo'].items()])

pm, ref, rob = R['por_metodo'], R['referencias'], R['robustez']
t5 = pd.DataFrame([
  ['Sin ajuste alguno',ms(ref['ATE_sin_ajuste']),'Referencia sesgada'],
  ['Ajuste con el DAG verdadero (oráculo)',ms(ref['ATE_dag_verdadero']),'Referencia insesgada'],
  ['Ajuste con el DAG del algoritmo genético',ms(pm['AG']['ATE']),'Coincide con el oráculo'],
  ['Ajuste con el DAG del Hill-Climbing',ms(pm['Hill-Climbing']['ATE']),'Sesgo moderado'],
  ['Ajuste con el DAG del PC',ms(pm['PC']['ATE']),'Sobreestimación'],
  ['Refutador: causa común aleatoria',ms(rob['causa_comun_aleatoria']),'Estable (aprueba 100 %)'],
  ['Refutador: tratamiento placebo',ms(rob['tratamiento_placebo']),'Colapsa a cero (aprueba 100 %)'],
  ['Refutador: submuestra al 80 %',ms(rob['submuestra_80']),'Estable (aprueba 100 %)'],
], columns=['Estimación o prueba','ATE','Interpretación'])

t6 = pd.DataFrame([['Sistema operativo',ENT['so']],
   ['Arquitectura / núcleos',f"{ENT['arquitectura']} / {ENT['nucleos_logicos']}"],
   ['Python',f"{ENT['python']} ({ENT['implementation']})"]]
   + [[k,v] for k,v in ENT['librerias'].items()], columns=['Elemento','Valor'])

mostrar('Tabla 1 — Comparación de enfoques de descubrimiento de estructura causal', t1)
mostrar('Tabla 2 — Hiperparámetros del algoritmo genético y del estimador', t2)
mostrar('Tabla 3 — Recuperación estructural por método (30 réplicas)', t3)
mostrar('Tabla 4 — Calidad del conjunto de ajuste', t4)
mostrar('Tabla 5 — Estimación del efecto causal y robustez', t5)
mostrar('Tabla 6 — Entorno de ejecución y versiones', t6)

## 7. Mostrar TODAS las figuras del documento

In [ ]:
import os
from IPython.display import Image, display, Markdown
figs = [('fig1_dags.png','Figura 1 — DAG verdadero vs. DAG recuperado por el AG'),
        ('fig2_convergencia.png','Figura 2 — Convergencia del algoritmo genético'),
        ('fig3_comparacion_metodos.png','Figura 3 — F1 del esqueleto por método'),
        ('fig4_distribucion_ate.png','Figura 4 — Distribución del efecto estimado')]
for archivo, titulo in figs:
    ruta = f'{PROJECT_DIR}/figures/{archivo}'
    display(Markdown(f'### {titulo}'))
    if os.path.exists(ruta):
        display(Image(ruta))
    else:
        print('(No encontrada — ejecuta antes la celda del Paso 5.)')

## 8. Descargar resultados y figuras a tu computadora

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/resultados_causal_ga_dml', 'zip', f'{PROJECT_DIR}/results')
shutil.make_archive('/content/figuras_causal_ga_dml', 'zip', f'{PROJECT_DIR}/figures')
files.download('/content/resultados_causal_ga_dml.zip')
files.download('/content/figuras_causal_ga_dml.zip')

---
Detalle de semillas, hiperparámetros y versiones: `docs/REPRODUCIBILITY.md`.
Formalización metodológica: `docs/METHODS.md`.